# KBM parallel extent against the physics parameters of the kbm_free hypercubes (Fusion_PhD-kurr.2)

**Question.** How does the parallel extent of GS2 KBMs depend on the eight sampled parameters (and on the derived $\beta'$ and $\alpha$)? Does a power law like the MTM one ($\theta_{95} \propto k_y^{a}\hat s^{b} q^{d}$, `results/mtm_extent_law_refit`) hold, which parameters matter, and are the exponents shared across the four equilibria? **No new runs; advisory.**

**Data.** The four free-parity GS2 Latin hypercubes `GS2/Runs/LATIN_HYPERCUBE/<case>/kbm_free` (SPR-045, M1, R3, R4; 1000 samples each, saved after the option-B reruns). Each `pyroscan.nc` is pyro's `time_mode="average"` reduction (last 20 % of the time trace), and holds the tearing parameter computed by pyro at save time.

**Extent.** pyrokinetics `Extent`: 95 % full span in $\theta$ (in $\pi$) of $|\phi|^2+|A_\parallel|^2$ from the cube's `eigenfunctions_squared` (the MTM law's field choice). $\phi$-only is reported beside it.

**Population.** User classifier (`general_analysis.mode_classification.classify`, docs/mode_filtering.md): KBM = $T<0.05$, $\omega>0$, $0.25<\chi_e/\chi_i<4$. Then $\gamma>0$ and converged (`growth_rate_tolerance` < 0.01). Box contact ($\ge 1\%$ of the power in $|\theta|>0.9\,\theta_{\max}$) is kept as **right-censored** (Tobit fit), not dropped.

**Parameters** are the cube coordinates. $\beta'$ and $\alpha=-q^2 R_0\beta'$ come from each sample's Pyro (`scan.sample_pyro`), after re-applying the run relations GyroRun used (see the next cell).

In [ ]:
import os, sys, json, warnings, re
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
from scipy import stats, optimize
from matplotlib.ticker import StrMethodFormatter
from dotenv import load_dotenv
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent
from pyrokinetics.pyroscan import PyroScanGKOutput

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from general_analysis.mode_classification import classify, ION_DIRECTION

warnings.filterwarnings("ignore")
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 12})  # PLOTTING_AND_PUBLISHING 4a
load_dotenv(ROOT / "local.env", override=True)
lhc = Path(os.environ["GK_DATA_ROOT"]).expanduser() / "GS2/Runs/LATIN_HYPERCUBE"
plot_dir = ROOT / "Plots" / "kbm_extent_vs_physics_parameters"
plot_dir.mkdir(parents=True, exist_ok=True)

databases = ["SPR-045", "M1", "R3", "R4"]
TOLERANCE_MAX, EDGE_MIN = 1e-2, 1e-2
sampled = ["ky", "q", "shat", "beta", "ion_temp_gradient", "electron_temp_gradient", "electron_dens_gradient", "electron_nu"]
derived = ["beta_prime", "alpha"]   # |beta'| and alpha = -q^2 R0 beta', from each sample's Pyro
symbol = dict(ky=r"$k_y\rho$", q="$q$", shat=r"$\hat s$", beta=r"$\beta$", ion_temp_gradient="$a/L_{Ti}$",
              electron_temp_gradient="$a/L_{Te}$", electron_dens_gradient="$a/L_{n}$", electron_nu=r"$\nu_e$",
              beta_prime=r"$|\beta'|$", alpha=r"$\alpha$")
colour = dict(zip(databases, ["C0", "C1", "C2", "C3"]))

def plain_ticks(ax):
    "Log axes: 1-2-5 ticks inside the limits, plain {x:g} labels, no minor ticks."
    for axis, lim, setter in ((ax.xaxis, ax.get_xlim(), ax.set_xticks), (ax.yaxis, ax.get_ylim(), ax.set_yticks)):
        if axis.get_scale() != "log": continue
        ticks = [m * 10.0 ** e for e in range(int(np.floor(np.log10(lim[0]))) - 1, int(np.ceil(np.log10(lim[1]))) + 1) for m in (1, 2, 5)]
        setter([v for v in ticks if lim[0] <= v <= lim[1]]); axis.set_major_formatter(StrMethodFormatter("{x:g}"))
        axis.set_minor_formatter(plt.NullFormatter()); axis.set_minor_locator(plt.NullLocator())
        (ax.set_xlim if axis is ax.xaxis else ax.set_ylim)(lim)

## Load

One `PyroHypercube` per database from its `pyroscan.json`, with its `pyroscan.nc` attached. Nothing is read from the run directories except in the provenance check below.

In [ ]:
scans = {}
for database in databases:
    leaf = lhc / database / "kbm_free"
    scans[database] = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    scans[database].load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    print(database, dict(scans[database].gk_output.data.sizes), "| species", scans[database].base_pyro.local_species.names)

## Per-sample physics from pyro

**The run relations are re-applied here.** GyroRun built these decks by setting the sampled values, then: every ion's $a/L_n$ = the electron's, every ion's $a/L_T$ = the first ion's, `enforce_quasineutrality("electron")`, `enforce_consistent_beta_prime()` (`results/kbm_gs2_1000/config_<case>.yaml`). It attached those as an unnamed function, which `pyroscan.json` does not store, so `sample_pyro` alone returns the base deck's ion gradients and $\beta'$. The same pyro calls are made below; the next cell checks them against the deck GS2 actually read (embedded in `input.out.nc`). Ion $\nu$ is not re-linked: it enters neither $\chi$ nor $\beta'$.

$\chi_s=(Q_s-\tfrac32 T_s\Gamma_s)/(n_sT_s\,a/L_{Ts})$, summed over the ions for $\chi_i$, as in `mode_classification.indicators`. Extent per sample after dropping the NaN padding of the union $\theta$ grid.

In [ ]:
rows = []
for database, scan in scans.items():
    output = scan.gk_output.data.pint.dequantify()
    heat, particle = output["heat"].sum("field"), output["particle"].sum("field")
    for index, sample_name in enumerate(np.asarray(output.sample_name.values, dtype=str)):
        pyro = scan.sample_pyro(index)
        species = pyro.local_species
        ions = [name for name in species.names if name != "electron"]
        for ion in ions:
            species[ion].inverse_ln = species.electron.inverse_ln
            species[ion].inverse_lt = species[ions[0]].inverse_lt
        species.enforce_quasineutrality("electron")
        pyro.enforce_consistent_beta_prime()

        def chi(group):
            transport = sum(float(heat.sel(sample=index, species=s)) - 1.5 * species[s].temp.m * float(particle.sel(sample=index, species=s)) for s in group)
            drive = sum(species[s].dens.m * species[s].temp.m * species[s].inverse_lt.m for s in group)
            return transport / drive

        eigenfunction_power = output["eigenfunctions_squared"].isel(sample=index, kx=0)
        both_fields = eigenfunction_power.sel(field=["phi", "apar"]).sum("field", skipna=False).dropna("theta")
        phi_only = eigenfunction_power.sel(field="phi").dropna("theta")
        extent = {}
        for name, power in (("extent", both_fields), ("extent_phi", phi_only)):
            if power.theta.size < 2:
                extent[name] = np.nan; continue
            wrapped = PyroScanGKOutput(xr.Dataset({"eigenfunctions_squared": power}))
            Extent(wrapped)
            extent[name] = float(wrapped["extent"]) / np.pi
        theta = both_fields.theta.values
        edge = float(np.trapezoid(both_fields.values * (abs(theta) > 0.9 * abs(theta).max()), theta) / np.trapezoid(both_fields.values, theta)) if theta.size > 1 else np.nan
        beta_prime = float(pyro.local_geometry.beta_prime.m)
        rows.append(dict(database=database, sample=sample_name, **{p: float(output[p][index]) for p in sampled},
                         beta_prime=abs(beta_prime), alpha=-float(pyro.local_geometry.q) ** 2 * float(pyro.local_geometry.Rmaj) * beta_prime,
                         gamma=float(output["growth_rate"][index]), omega=ION_DIRECTION * float(output["mode_frequency"][index]),
                         T=float(output["tearing_parameter"][index]), tolerance=float(output["growth_rate_tolerance"][index]),
                         chi_ratio=chi(["electron"]) / chi(ions), edge=edge, **extent))
samples = pd.DataFrame(rows)
samples["alpha"] = samples["alpha"].astype(float)
samples["label"] = classify(samples)
print(samples.groupby("database", sort=False).label.value_counts().unstack().to_string())

### Provenance check: the re-applied relations against the deck GS2 read

Three samples per database: $\beta'$ (`beta_prime_input`) and every species' `tprim`/`fprim` parsed from the input file that GS2 stores in `input.out.nc`.

In [ ]:
for database, scan in scans.items():
    for sample_name in np.asarray(scan.gk_output.data.sample_name.values, dtype=str)[[3, 400, 900]]:
        with xr.open_dataset(lhc / database / "kbm_free" / sample_name / "input.out.nc") as run_output:
            deck = "\n".join(b"".join(line).decode().strip() for line in run_output["input_file"].values)
        deck_beta_prime = float(re.search(r"beta_prime_input\s*=\s*([-\d.eE+]+)", deck).group(1))
        deck_gradients = sorted(float(v) for v in re.findall(r"^\s*[tf]prim\s*=\s*([-\d.eE+]+)", deck, re.M))
        row = samples[(samples.database == database) & (samples["sample"] == sample_name)].iloc[0]
        pyro = scan.sample_pyro(sample_name)
        species = pyro.local_species
        ions = [name for name in species.names if name != "electron"]
        for ion in ions:
            species[ion].inverse_ln = species.electron.inverse_ln
            species[ion].inverse_lt = species[ions[0]].inverse_lt
        species.enforce_quasineutrality("electron")
        pyro_gradients = sorted([float(species[s].inverse_lt.m) for s in species.names] + [float(species[s].inverse_ln.m) for s in species.names])
        print(f"{database} {sample_name}: beta' deck {deck_beta_prime:+.6f} notebook {-row.beta_prime:+.6f} | gradients max |diff| {np.max(np.abs(np.array(deck_gradients) - pyro_gradients)):.1e}")

## Population and counts

Drops in order: not labelled KBM, $\gamma\le0$, not converged (tolerance $\ge 0.01$ or NaN). Survivors split into **kept** (edge < 1e-2) and **censored** (box contact; the extent is a lower bound).

In [ ]:
unstable_kbm = (samples.label == "KBM") & (samples.gamma > 0)
converged = samples.tolerance < TOLERANCE_MAX
population = samples[unstable_kbm & converged & samples.extent.notna()].assign(censored=lambda d: d.edge >= EDGE_MIN)
counts = pd.DataFrame({
    "loaded": samples.groupby("database").size(), "gamma finite": samples[samples.gamma.notna()].groupby("database").size(),
    "KBM": samples[samples.label == "KBM"].groupby("database").size(), "KBM gamma>0": samples[unstable_kbm].groupby("database").size(),
    "converged": population.groupby("database").size(), "kept": population[~population.censored].groupby("database").size(),
    "censored": population[population.censored].groupby("database").size(),
    "median tolerance (KBM gamma>0)": samples[unstable_kbm].groupby("database").tolerance.median(),
}).reindex(databases).fillna(0)
print(counts.to_string())

## Fits

$\ln\theta_{95} = c_{\rm db} + \sum_j a_j \ln p_j$ over the eight sampled parameters. **Tobit** (right-censored Gaussian likelihood: a censored mode contributes $P(\ln\theta > \text{measured})$). 95 % CI from a 200-sample bootstrap. Drop-one $\Delta$AIC = AIC without that parameter − AIC with all; $>0$ means the parameter is needed. Per database, and pooled with one intercept per database (shared exponents, surface-specific prefactor).

In [ ]:
def design(data, names, intercepts=("all",)):
    columns = [np.ones(len(data))] if intercepts == ("all",) else [(data.database == db).values.astype(float) for db in intercepts]
    return np.column_stack(columns + [np.log(data[name].values) for name in names])

def tobit_fit(design_matrix, log_extent, censored, start):
    def negative_log_likelihood(parameters):
        z = (log_extent - design_matrix @ parameters[:-1]) / np.exp(parameters[-1])
        return -(stats.norm.logpdf(z[~censored]) - parameters[-1]).sum() - stats.norm.logsf(z[censored]).sum()
    result = optimize.minimize(negative_log_likelihood, start, method="BFGS")
    return result.x, result.fun

def tobit(data, names, intercepts=("all",), n_boot=200, seed=0):
    design_matrix, log_extent, censored = design(data, names, intercepts), np.log(data.extent.values), data.censored.values
    start = np.r_[np.linalg.lstsq(design_matrix, log_extent, rcond=None)[0], np.log(np.std(log_extent))]
    best, negative_log_likelihood = tobit_fit(design_matrix, log_extent, censored, start)
    fit = dict(names=names, n_intercepts=len(intercepts), b=best[:-1], sigma=float(np.exp(best[-1])), n=len(data),
               n_censored=int(censored.sum()), aic=2 * len(best) + 2 * negative_log_likelihood, loglik=-negative_log_likelihood)
    if n_boot:
        rng = np.random.default_rng(seed)
        boot = np.array([tobit_fit(design_matrix[i], log_extent[i], censored[i], best)[0]
                         for i in (rng.integers(0, len(log_extent), len(log_extent)) for _ in range(n_boot))])
        low, high = np.percentile(boot[:, :-1], [2.5, 97.5], axis=0)
        fit["ci"] = (high - low) / 2
    return fit

def slopes(fit):
    return dict(zip(fit["names"], fit["b"][fit["n_intercepts"]:]))

fits, drop_one = {}, {}
for group in databases + ["pooled"]:
    data = population if group == "pooled" else population[population.database == group]
    intercepts = tuple(databases) if group == "pooled" else ("all",)
    fits[group] = tobit(data, sampled, intercepts)
    drop_one[group] = {name: tobit(data, [p for p in sampled if p != name], intercepts, n_boot=0)["aic"] - fits[group]["aic"] for name in sampled}
    ci = dict(zip(sampled, fits[group]["ci"][fits[group]["n_intercepts"]:]))
    print(f"{group} (n={fits[group]['n']}, censored {fits[group]['n_censored']}, sigma {fits[group]['sigma']:.3f}):")
    print("   " + "  ".join(f"{name} {slope:+.3f}±{ci[name]:.3f} [dAIC {drop_one[group][name]:+.1f}]" for name, slope in slopes(fits[group]).items()))

## Are the exponents shared across the four databases? Does a power law hold?

**Sharing.** Likelihood-ratio test of the pooled fit (one intercept per database, shared exponents) against the four separate fits, and, parameter by parameter, of letting only that exponent differ by database (3 extra degrees of freedom).
**Power law.** For each parameter, $\Delta$AIC of adding $(\ln p)^2$ to the pooled fit: a large drop means the dependence curves in log-log, so a single exponent misdescribes it.

In [ ]:
pooled = fits["pooled"]
separate_loglik = sum(fits[db]["loglik"] for db in databases)
extra_df = sum(len(fits[db]["b"]) + 1 for db in databases) - (len(pooled["b"]) + 1)
lr_all = 2 * (separate_loglik - pooled["loglik"])
print(f"all exponents shared vs separate: LR {lr_all:.1f} on {extra_df} df, p = {stats.chi2.sf(lr_all, extra_df):.2e}")
sharing, curvature = {}, {}
for name in sampled:
    interacted = population.assign(**{f"{name}@{db}": np.where(population.database == db, population[name], 1.0) for db in databases[1:]})
    fit_interacted = tobit(interacted, sampled + [f"{name}@{db}" for db in databases[1:]], tuple(databases), n_boot=0)
    lr = 2 * (fit_interacted["loglik"] - pooled["loglik"])
    sharing[name] = dict(LR=lr, p=stats.chi2.sf(lr, len(databases) - 1),
                         per_database={db: fits[db]["b"][1 + sampled.index(name)] for db in databases})
    squared = population.assign(**{f"{name}_sq": np.exp(np.log(population[name]) ** 2)})
    curvature[name] = tobit(squared, sampled + [f"{name}_sq"], tuple(databases), n_boot=0)["aic"] - pooled["aic"]
    print(f"{name:24s} sharing LR {lr:6.1f} p={sharing[name]['p']:.1e} | per db " + " ".join(f"{v:+.2f}" for v in sharing[name]["per_database"].values())
          + f" | add (ln p)^2: dAIC {curvature[name]:+.1f}")

## Derived parameters: $|\beta'|$ and $\alpha$

$\beta'$ and $\alpha$ are functions of the sampled axes ($\beta$, the gradients, $q$), so they cannot enter the joint fit beside them. Two checks: (1) the pooled fit with $\beta$ replaced by $\alpha$ (AIC against the sampled-axis fit); (2) the partial slope of $\ln\alpha$ on the residual of the full sampled-axis fit (what $\alpha$ explains that the axes do not).

In [ ]:
derived_fits = {}
for name in derived:
    alternative = tobit(population, [p if p != "beta" else name for p in sampled], tuple(databases))
    derived_fits[name] = alternative
    ci = dict(zip(alternative["names"], alternative["ci"][len(databases):]))
    print(f"beta -> {name}: AIC {alternative['aic'] - pooled['aic']:+.1f} vs sampled-axis fit; {name} exponent {slopes(alternative)[name]:+.3f}±{ci[name]:.3f}")
kept = population[~population.censored]
pooled_prediction = design(kept, sampled, tuple(databases)) @ pooled["b"]
for name in derived:
    fit_line = stats.linregress(np.log(kept[name]), np.log(kept.extent) - pooled_prediction)
    print(f"partial slope of residual on ln {name} (kept, n={len(kept)}): {fit_line.slope:+.3f}±{1.96 * fit_line.stderr:.3f}")

## Verdict table

Pooled Tobit exponent (shared, one prefactor per database) with 95 % bootstrap CI, drop-one $\Delta$AIC, and the implied change of $\theta_{95}$ from the 5th to the 95th percentile of the parameter in the population. "Depends" = CI excludes 0 and drop-one $\Delta$AIC > 10.

In [ ]:
verdict_rows = []
for name in sampled:
    slope, ci = slopes(pooled)[name], pooled["ci"][len(databases) + sampled.index(name)]
    p5, p95 = np.percentile(population[name], [5, 95])
    verdict_rows.append(dict(parameter=name, slope=slope, ci=ci, n=pooled["n"], dAIC_drop=drop_one["pooled"][name],
                             change_p5_p95=(p95 / p5) ** slope - 1, shared_p=sharing[name]["p"], dAIC_curvature=curvature[name],
                             verdict="depends" if abs(slope) > ci and drop_one["pooled"][name] > 10 else "no dependence found"))
verdicts = pd.DataFrame(verdict_rows)
print(verdicts.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

## Figure 1. Extent against each parameter (marginal)

Every KBM in the population; circles kept, triangles box contact (lower bounds). Colour = database.

In [ ]:
panels = sampled + derived
fig, axes = plt.subplots(2, 5, figsize=(22, 9), sharey=True)
for ax, name in zip(axes.flat, panels):
    for database in databases:
        data = population[population.database == database]
        ax.scatter(data[name][~data.censored], data.extent[~data.censored], s=8, alpha=0.5, c=colour[database], label=database)
        ax.scatter(data[name][data.censored], data.extent[data.censored], s=18, marker="^", alpha=0.7, c=colour[database])
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel(symbol[name]); plain_ticks(ax)
for ax in axes[:, 0]: ax.set_ylabel(r"$\theta_{95}$ [$\pi$]")
axes[0, 0].legend(markerscale=2)
fig.savefig(plot_dir / "fig1_extent_vs_each_parameter.png")

## Figure 2. Partial residuals (pooled fit)

$\ln\theta_{95}$ minus the database intercept and every other parameter's term, against each parameter. Line: the fitted term.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(19, 9), sharey=True)
intercept_by_database = dict(zip(databases, pooled["b"][:len(databases)]))
full_prediction = design(population, sampled, tuple(databases)) @ pooled["b"]
for ax, name in zip(axes.flat, sampled):
    term = slopes(pooled)[name] * np.log(population[name])
    partial = np.log(population.extent) - full_prediction + term
    for database in databases:
        mask = population.database == database
        ax.scatter(population[name][mask & ~population.censored], partial[mask & ~population.censored], s=8, alpha=0.5, c=colour[database], label=database)
        ax.scatter(population[name][mask & population.censored], partial[mask & population.censored], s=18, marker="^", alpha=0.7, c=colour[database])
    grid = np.geomspace(population[name].min(), population[name].max(), 50)
    ax.plot(grid, slopes(pooled)[name] * np.log(grid), "k", lw=1.5, marker="")
    ax.set_xscale("log"); ax.set_xlabel(symbol[name]); ax.set_title(f"{slopes(pooled)[name]:+.2f}"); plain_ticks(ax)
for ax in axes[:, 0]: ax.set_ylabel(r"partial residual of $\ln\theta_{95}$")
axes[0, 0].legend(markerscale=2)
fig.savefig(plot_dir / "fig2_partial_residuals.png")

## Figure 3. Measured extent against the pooled power law, per database

y = x is the fitted centre (shared exponents, that database's prefactor). Triangles: lower bounds.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True, sharey=True)
centre = np.exp(full_prediction)
limits = np.array([0.5 * min(centre.min(), population.extent.min()), 2 * max(centre.max(), population.extent.max())])
for ax, database in zip(axes.flat, databases):
    mask = (population.database == database).values
    ax.scatter(centre[mask & ~population.censored.values], population.extent[mask & ~population.censored.values], s=10, alpha=0.5, c=colour[database])
    ax.scatter(centre[mask & population.censored.values], population.extent[mask & population.censored.values], s=22, marker="^", alpha=0.8, c="k")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(limits); ax.set_ylim(limits)
    ax.plot(limits, limits, "k", lw=1, marker=""); ax.set_title(database); plain_ticks(ax)
for ax in axes[1]: ax.set_xlabel(r"power law $\theta_{95}$ [$\pi$]")
for ax in axes[:, 0]: ax.set_ylabel(r"measured $\theta_{95}$ [$\pi$]")
fig.savefig(plot_dir / "fig3_measured_vs_power_law.png")

## $\phi$-only extent

Same population; how far the $\phi$-only extent differs from $|\phi|^2+|A_\parallel|^2$, and the pooled fit on it (censoring flag unchanged).

In [ ]:
ratio = population.extent_phi / population.extent
print("phi-only / (phi+apar) extent: median", round(float(ratio.median()), 4), "p5-p95", np.round(np.percentile(ratio, [5, 95]), 4))
phi_fit = tobit(population.assign(extent=population.extent_phi), sampled, tuple(databases))
print("pooled exponents, phi only:", "  ".join(f"{k} {v:+.3f}" for k, v in slopes(phi_fit).items()))

In [ ]:
summary = dict(counts=counts.reset_index().rename(columns={"index": "database"}).to_dict("records"),
               fits={g: dict(slopes=slopes(f), ci=dict(zip(sampled, map(float, f["ci"][f["n_intercepts"]:]))),
                             prefactor=dict(zip(databases if g == "pooled" else [g], map(float, np.exp(f["b"][:f["n_intercepts"]])))),
                             sigma=f["sigma"], n=f["n"], n_censored=f["n_censored"], drop_one_dAIC=drop_one[g]) for g, f in fits.items()},
               sharing=dict(all=dict(LR=lr_all, df=extra_df, p=stats.chi2.sf(lr_all, extra_df)), per_parameter=sharing), curvature_dAIC=curvature,
               derived={name: dict(dAIC_vs_beta=f["aic"] - pooled["aic"], exponent=slopes(f)[name]) for name, f in derived_fits.items()},
               verdicts=verdicts.to_dict("records"), phi_only=dict(median_ratio=float(ratio.median()), slopes=slopes(phi_fit)))
json.dump(summary, open(plot_dir / "summary.json", "w"), indent=1, default=float)